# Notebook 7T — Full Hyperparameter Tuning of All Five Baselines (Table 3)

**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

This notebook tunes **every** baseline model (not just the winner) so Table 3 can
report a genuine **before vs after** comparison for all five.

Design, so the table is a fair like-for-like:
- Each model is tuned on its **own baseline feature set (FS1 = TF-IDF unigram+bigram)** —
  the same features it used in Experiments 1-5. 'Before' is that model's baseline;
  'After' is the same model with tuned hyperparameters on the same features.
- Both **cross-validation macro F1** (the selection basis) and **test macro F1**
  are reported, so any case where tuning improves CV but not test is visible.
- **Random Forest is slow** (already ~56 min untuned). Its grid is kept deliberately
  small; expect this notebook to be dominated by the RF cell. Run others first.

The LR-on-hybrid tuning (Experiment 10) remains a separate result; this notebook is
about whether tuning changes each *baseline*, and whether it changes the ranking.


## Cell 1: Setup

In [1]:
!pip install -q emoji xgboost scikit-learn pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *

import pandas as pd, numpy as np, scipy.sparse as sp, pickle, time, json
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import f1_score, accuracy_score

D = PATHS["data"]
tw_train = pd.read_parquet(D/"tw_train.parquet")
tw_test  = pd.read_parquet(D/"tw_test.parquet")
X_train  = sp.load_npz(D/"tw_Xtrain_fs1.npz")
X_test   = sp.load_npz(D/"tw_Xtest_fs1.npz")
y_train  = tw_train["sentiment"].values
y_test   = tw_test["sentiment"].values
sg_test  = tw_test["subgroup_primary"].values
print("Train", X_train.shape, "Test", X_test.shape)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 11.5 MB/s eta 0:00:00
Mounted at /content/drive
thesis_utils loaded. [V2 FIXED: Fairlearn EOD, Theil index]
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal
Train (45615, 50000) Test (12284, 50000)


## Cell 2: Tuning runner

For one model: records the baseline (before) CV + test macro F1, runs GridSearchCV,
then records the tuned (after) CV + test macro F1. Saves tuned predictions so they
can be reused, exactly like every other experiment.

In [2]:
def tune_model(name, base_estimator, param_grid, needs_encoding=False,
               base_params_label="default", cv_override=5):
    from sklearn.base import clone
    print("="*64); print(f"TUNING — {name}"); print("="*64)

    le = None
    if needs_encoding:
        from sklearn.preprocessing import LabelEncoder
        le = LabelEncoder().fit(y_train)
        ytr = le.transform(y_train)
    else:
        ytr = y_train

    # ---- BEFORE: baseline estimator, CV + test ----
    base = clone(base_estimator)
    t0=time.time()
    cv_before = cross_val_score(base, X_train, ytr, cv=cv_override,
                                scoring="f1_macro", n_jobs=-1).mean()
    base.fit(X_train, ytr)
    yp_before = base.predict(X_test)
    if needs_encoding: yp_before = le.inverse_transform(yp_before)
    test_before = f1_score(y_test, yp_before, average="macro", zero_division=0)
    acc_before  = accuracy_score(y_test, yp_before)

    # ---- AFTER: grid search, CV + test ----
    gs = GridSearchCV(clone(base_estimator), param_grid, cv=cv_override,
                      scoring="f1_macro", n_jobs=-1)
    gs.fit(X_train, ytr)
    cv_after = gs.best_score_
    yp_after = gs.best_estimator_.predict(X_test)
    if needs_encoding: yp_after = le.inverse_transform(yp_after)
    test_after = f1_score(y_test, yp_after, average="macro", zero_division=0)
    acc_after  = accuracy_score(y_test, yp_after)
    elapsed=(time.time()-t0)/60

    # save tuned predictions for reuse
    proba = gs.best_estimator_.predict_proba(X_test)
    save_predictions(f"exp_tune_{name}", name, y_test, yp_after, proba, sg_test)

    row = {
        # ---- EXACT SHEET COLUMNS (paste straight into Table 3) ----
        "Model": name,
        "Dataset": "TweetEval",
        "Feature Set Used": "TF-IDF unigram + bigram (FS1)",
        "Before Tuning Accuracy": round(acc_before,4),
        "Before Tuning Macro F1": round(test_before,4),
        "After Tuning Accuracy": round(acc_after,4),
        "After Tuning Macro F1": round(test_after,4),
        "Best Parameters": str(gs.best_params_),
        "Macro F1 Improvement": round(test_after-test_before,4),
        # "Final Rank After Tuning" added after sorting, below
        # ---- APPENDED EXTRA COLUMNS (CV basis + timing) ----
        "Before Tuning CV Macro F1": round(cv_before,4),
        "After Tuning CV Macro F1": round(cv_after,4),
        "CV Macro F1 Change": round(cv_after-cv_before,4),
        "Tuning Time (min)": round(elapsed,1),
    }
    print(f"  CV   before {cv_before:.4f} -> after {cv_after:.4f}  (Δ {cv_after-cv_before:+.4f})")
    print(f"  Test before {test_before:.4f} -> after {test_after:.4f}  (Δ {test_after-test_before:+.4f})")
    print(f"  Acc  before {acc_before:.4f} -> after {acc_after:.4f}")
    print(f"  best params: {gs.best_params_}   time {elapsed:.1f} min")

    # --- SAVE THIS MODEL IMMEDIATELY so a Colab disconnect never loses completed work ---
    per_model_dir = PATHS["results"] / "tuning_partial"
    per_model_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row]).to_csv(per_model_dir / f"tune_{name}.csv", index=False)
    print(f"  saved -> tuning_partial/tune_{name}.csv  (safe against disconnect)")
    return row

tuning_rows = []


In [3]:
import pandas as pd
from pathlib import Path
PARTIAL = PATHS["results"] / "tuning_partial"
PARTIAL.mkdir(parents=True, exist_ok=True)

def run_or_skip(name, *args, **kwargs):
    """Run tune_model(name, ...) unless its result already exists on disk.
    Lets you resume after a Colab disconnect without redoing finished models."""
    f = PARTIAL / f"tune_{name}.csv"
    if f.exists():
        print(f"SKIP {name} - already saved. Delete tuning_partial/tune_{name}.csv to re-run.")
        return pd.read_csv(f).iloc[0].to_dict()
    return tune_model(name, *args, **kwargs)

print("resume helper ready - finished models are skipped on re-run")

resume helper ready - finished models are skipped on re-run


## Cell 3: Logistic Regression

In [4]:
from sklearn.linear_model import LogisticRegression
tuning_rows.append(run_or_skip(
    "LogisticRegression",
    LogisticRegression(max_iter=2000, random_state=SEED, solver="liblinear"),
    {"C":[0.01,0.1,1,10], "penalty":["l1","l2"]},
))

TUNING — LogisticRegression
  saved predictions -> exp_tune_LogisticRegression_LogisticRegression_TweetEval.parquet  (12,284 rows)
  CV   before 0.5840 -> after 0.6158  (Δ +0.0319)
  Test before 0.5453 -> after 0.5672  (Δ +0.0219)
  Acc  before 0.5865 -> after 0.5800
  best params: {'C': 10, 'penalty': 'l2'}   time 1.4 min
  saved -> tuning_partial/tune_LogisticRegression.csv  (safe against disconnect)


## Cell 4: Linear SVM (calibrated for probabilities)

In [5]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
tuning_rows.append(run_or_skip(
    "LinearSVM",
    CalibratedClassifierCV(LinearSVC(max_iter=3000, random_state=SEED, dual="auto"),
                           cv=3, method="sigmoid"),
    {"estimator__C":[0.01,0.1,1,10]},
))

TUNING — LinearSVM
  saved predictions -> exp_tune_LinearSVM_LinearSVM_TweetEval.parquet  (12,284 rows)
  CV   before 0.6097 -> after 0.6097  (Δ +0.0000)
  Test before 0.5648 -> after 0.5648  (Δ +0.0000)
  Acc  before 0.5876 -> after 0.5876
  best params: {'estimator__C': 1}   time 1.2 min
  saved -> tuning_partial/tune_LinearSVM.csv  (safe against disconnect)


## Cell 5: Multinomial Naive Bayes (fast)

In [6]:
from sklearn.naive_bayes import MultinomialNB
tuning_rows.append(run_or_skip(
    "MultinomialNB",
    MultinomialNB(),
    {"alpha":[0.1,0.5,1.0,2.0]},
))

TUNING — MultinomialNB
  saved predictions -> exp_tune_MultinomialNB_MultinomialNB_TweetEval.parquet  (12,284 rows)
  CV   before 0.4702 -> after 0.5846  (Δ +0.1144)
  Test before 0.4349 -> after 0.5589  (Δ +0.1241)
  Acc  before 0.5339 -> after 0.5791
  best params: {'alpha': 0.1}   time 0.1 min
  saved -> tuning_partial/tune_MultinomialNB.csv  (safe against disconnect)


## Cell 6: XGBoost (fast-ish)

In [7]:
# XGBoost — GPU-accelerated (falls back to CPU if no GPU is attached).
# In Colab: Runtime -> Change runtime type -> Hardware accelerator: GPU (T4 is fine).
from xgboost import XGBClassifier
import subprocess
def _gpu_available():
    try:
        subprocess.check_output(["nvidia-smi"]); return True
    except Exception:
        return False
GPU = _gpu_available()
print("GPU detected:" , GPU)

xgb_kwargs = dict(random_state=SEED, n_jobs=-1, eval_metric="mlogloss")
if GPU:
    # newer xgboost API: device="cuda"; older: tree_method="gpu_hist"
    try:
        xgb_kwargs.update(device="cuda", tree_method="hist")
    except Exception:
        xgb_kwargs.update(tree_method="gpu_hist")
else:
    xgb_kwargs.update(tree_method="hist")

tuning_rows.append(run_or_skip(
    "XGBoost",
    XGBClassifier(**xgb_kwargs),
    {"n_estimators":[200,400], "learning_rate":[0.1,0.3], "max_depth":[5,7]},
    needs_encoding=True,
))

GPU detected: True
TUNING — XGBoost
  saved predictions -> exp_tune_XGBoost_XGBoost_TweetEval.parquet  (12,284 rows)
  CV   before 0.5639 -> after 0.5965  (Δ +0.0326)
  Test before 0.4671 -> after 0.5230  (Δ +0.0559)
  Acc  before 0.5602 -> after 0.5825
  best params: {'learning_rate': 0.3, 'max_depth': 7, 'n_estimators': 400}   time 40.3 min
  saved -> tuning_partial/tune_XGBoost.csv  (safe against disconnect)


## Cell 7: Random Forest — SLOW

RF was ~56 min untuned. This grid is intentionally tiny (4 combinations) to keep it
bounded, and it will still likely take a few hours. Run this cell last / overnight.
If it is too slow, reduce n_estimators to [200] and min_samples_leaf to [1,5].

In [8]:
# Random Forest — sklearn RF is CPU-ONLY (GPU cannot accelerate it).
# The time cost is the grid size, so this grid is kept session-safe: 2 combinations x 3-fold = 6 fits.
# This still constitutes real hyperparameter tuning. If you have Colab Pro / lots of time,
# you may widen to n_estimators:[200,400], min_samples_leaf:[1,5] and cv=5.
from sklearn.ensemble import RandomForestClassifier
tuning_rows.append(run_or_skip(
    "RandomForest",
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    {"n_estimators":[200], "min_samples_leaf":[1,5]},   # 2 combinations
    cv_override=3,                                        # 3-fold to bound time
))

TUNING — RandomForest
  saved predictions -> exp_tune_RandomForest_RandomForest_TweetEval.parquet  (12,284 rows)
  CV   before 0.4893 -> after 0.4893  (Δ -0.0000)
  Test before 0.4169 -> after 0.4169  (Δ +0.0001)
  Acc  before 0.5383 -> after 0.5394
  best params: {'min_samples_leaf': 1, 'n_estimators': 200}   time 32.6 min
  saved -> tuning_partial/tune_RandomForest.csv  (safe against disconnect)


## Cell 8: Table 3 — full before/after tuning for all five models

In [9]:
partial_files = sorted((PATHS["results"]/"tuning_partial").glob("tune_*.csv"))
tuning_rows = [pd.read_csv(f).iloc[0].to_dict() for f in partial_files]
print(f"loaded {len(tuning_rows)} completed models: {[r['Model'] for r in tuning_rows]}")
table3 = pd.DataFrame(tuning_rows)
# rank by tuned (after) test macro F1 — the sheet's headline metric
table3 = table3.sort_values("After Tuning Macro F1", ascending=False).reset_index(drop=True)
table3["Final Rank After Tuning"] = range(1, len(table3)+1)

# add the Gradient Boosting row the sheet template lists (never run: boosting model is XGBoost)
gb = {"Model":"Gradient Boosting","Dataset":"TweetEval","Feature Set Used":"TF-IDF unigram + bigram (FS1)",
      "Best Parameters":"not run — boosting baseline in this study is XGBoost",
      "Final Rank After Tuning":""}
table3 = pd.concat([table3, pd.DataFrame([gb])], ignore_index=True)

# EXACT sheet column order first, then appended extras
sheet_cols = ["Model","Dataset","Feature Set Used",
              "Before Tuning Accuracy","Before Tuning Macro F1",
              "After Tuning Accuracy","After Tuning Macro F1",
              "Best Parameters","Macro F1 Improvement","Final Rank After Tuning"]
extra_cols = ["Before Tuning CV Macro F1","After Tuning CV Macro F1","CV Macro F1 Change","Tuning Time (min)"]
table3 = table3[[c for c in sheet_cols+extra_cols if c in table3.columns]]

print("="*110)
print("TABLE 3 — HYPERPARAMETER TUNING (ALL FIVE BASELINES ON FS1)")
print("="*110)
print(table3[sheet_cols].to_string(index=False))
print("\n--- appended extra columns (CV basis + timing) ---")
print(table3[["Model"]+extra_cols].to_string(index=False))

save_result_table(table3, "Table3_Full_Tuning_All_Models")
print("\nNOTE: LR here is tuned on FS1 (shared features, fair cross-model comparison).")
print("The FINAL model is LR on the HYBRID set (Experiment 10, macro F1 0.5878), reported there.")
print("Gradient Boosting row is template-only; the executed boosting model is XGBoost.")


loaded 5 completed models: ['LinearSVM', 'LogisticRegression', 'MultinomialNB', 'RandomForest', 'XGBoost']
TABLE 3 — HYPERPARAMETER TUNING (ALL FIVE BASELINES ON FS1)
             Model   Dataset              Feature Set Used  Before Tuning Accuracy  Before Tuning Macro F1  After Tuning Accuracy  After Tuning Macro F1                                             Best Parameters  Macro F1 Improvement Final Rank After Tuning
LogisticRegression TweetEval TF-IDF unigram + bigram (FS1)                  0.5865                  0.5453                 0.5800                 0.5672                                  {'C': 10, 'penalty': 'l2'}                0.0219                       1
         LinearSVM TweetEval TF-IDF unigram + bigram (FS1)                  0.5876                  0.5648                 0.5876                 0.5648                                         {'estimator__C': 1}                0.0000                       2
     MultinomialNB TweetEval TF-IDF unigram + bigram (FS

## Cell 9: Done

`Table3_Full_Tuning_All_Models.csv` now has a genuine before/after for every model.
Tuned predictions saved as `exp_tune_<model>` for reuse. Note whether tuning changed
the model ranking; if the best model is unchanged, that is itself a reportable finding
(the baseline choice was robust to tuning).

In [10]:
print("NB7T complete — full tuning table generated.")

NB7T complete — full tuning table generated.
